In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt

In [ ]:
# ==========================================
# 1. Read and preprocess the data
# ==========================================
print("Reading data...")
try:
    df = pd.read_csv('../data/BTCUSD1D.csv')
except FileNotFoundError:
    print("Error: File 'BTCUSD1D.csv' not found.")
    exit()

In [ ]:
# Convert datetime column to datetime format and sort chronologically
df['datetime'] = pd.to_datetime(df['datetime'])
df = df.sort_values('datetime')
df = df.set_index('datetime')

In [ ]:
# ==========================================
# 2. Create Features (X) and Target (y)
# ==========================================
# Use the 'close' prices of 30 consecutive sessions (t-1 to t-30) to predict session t
window_size = 30

In [ ]:
print(f"Creating data with a window of {window_size} sessions...")
for i in range(1, window_size + 1):
    df[f'Lag_{i}'] = df['close'].shift(i)

In [ ]:
# Drop rows with missing data (NaN) caused by the shift operation
df = df.dropna()

In [ ]:
# Define feature columns (Lag_1 to Lag_30) and target column
feature_cols = [f'Lag_{i}' for i in range(1, window_size + 1)]
X = df[feature_cols]
y = df['close']

In [ ]:
# ==========================================
# 3. Split Train / Test sets
# ==========================================
split_date = '2026-03-01'

In [ ]:
X_train = X.loc[X.index < split_date]
y_train = y.loc[y.index < split_date]

In [ ]:
X_test = X.loc[X.index >= split_date]
y_test = y.loc[y.index >= split_date]

In [ ]:
print(f"Number of Train samples: {len(X_train)}")
print(f"Number of Test samples: {len(X_test)}")

In [ ]:
if len(X_train) == 0 or len(X_test) == 0:
    print("Error: Not enough data for Train/Test split based on the selected date.")
    exit()

In [ ]:
# ==========================================
# 4. Train the XGBoost Regressor model
# ==========================================
print("Training the XGBoost model...")
# Initialize XGBoost Regressor with basic parameters
model = xgb.XGBRegressor(
    n_estimators=500,      # Number of trees
    learning_rate=0.05,     # Step size shrinkage
    max_depth=8,           # Maximum depth of a tree
    objective='reg:squarederror'
)

In [ ]:
model.fit(X_train, y_train)

In [ ]:
# ==========================================
# 5. Evaluate the model
# ==========================================
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

In [ ]:
train_r2 = r2_score(y_train, y_pred_train)
test_r2 = r2_score(y_test, y_pred_test)
mae = mean_absolute_error(y_test, y_pred_test)

In [ ]:
print(f"R^2 accuracy on Train set: {train_r2:.4f}")
print(f"R^2 accuracy on Test set:  {test_r2:.4f}")
print(f"Mean Absolute Error (MAE) on Test set: {mae:.2f}")

In [ ]:
# ==========================================
# 6. Display Predicted vs Actual Results
# ==========================================
print("\nPrediction results from 2026-03-01 onwards:")
results_df = pd.DataFrame({
    'Actual Price': y_test, 
    'Predicted Price': y_pred_test
})
# Format the output slightly for cleaner display
print(results_df.round(2).head(20))

In [ ]:
# ==========================================
# 7. Visualize predicted and actual prices
# ==========================================
plt.figure(figsize=(12, 6))
plt.plot(results_df.index, results_df['Actual Price'], label='Actual Price', color='blue', alpha=0.7)
plt.plot(results_df.index, results_df['Predicted Price'], label='Predicted Price', color='red', linestyle='--', alpha=0.7)
plt.title('BTC Price Prediction (XGBoost) - 30 Recent Sessions')
plt.xlabel('Date')
plt.ylabel('Price (USD)')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# ==========================================
# 8. Save the model for reuse
# ==========================================
# XGBoost has its own optimized method for saving models (JSON format is recommended)
model_filename = 'models/3_xgboost_btc_30days_model.json'
model.save_model(model_filename)
print(f"\nModel successfully saved to file: '{model_filename}'")

In [ ]:
# ==========================================
# 9. Example: Predict the next session's price
# ==========================================
if not df.empty:
    # Get the 30 most recent 'close' values, reverse them to match the Lag_1 to Lag_30 order
    recent_30_days_features = df['close'].iloc[-window_size:].values[::-1].reshape(1, -1)
    
    # Create a DataFrame for the prediction feature matching the training column names
    recent_df = pd.DataFrame(recent_30_days_features, columns=feature_cols)

    # Load the model (simulating usage in another script/environment)
    loaded_model = xgb.XGBRegressor()
    loaded_model.load_model(model_filename)

    next_session_pred = loaded_model.predict(recent_df)
    print(f"\n--> Predicted BTC price for the next session based on the last 30 sessions is: {next_session_pred[0]:.2f}")